# Data Collection Portion of Project (API call)

### Need all the same python libraries and other tools for reproduciblity purposes, hence we store all of them in one .txt file and install them here. 

In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 6.0 MB/s eta 0:00:000:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 14.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: python-dotenv
    Found existing installation: python-dotenv 1.1.1
    Uninstalling python-dotenv-1.1.1:
      Successfully uninstalled python-dotenv-1.1.1
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


### Importing all the necessary modules / libraries for this notebook and also most importantly calling our API key that we hid in "secret_key.env" file which is not in Github Repo for security purposes. 

In [1]:
import os
from pathlib import Path
import pandas as pd
from fredapi import Fred
from dotenv import load_dotenv

load_dotenv("secret_key.env")
FRED_API_KEY = os.getenv("FRED_API_KEY")
assert FRED_API_KEY, "Set FRED_API_KEY in a .env file."
fred = Fred(api_key=FRED_API_KEY)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
START = "1990-01-01"

### Getting all the series from FredAPI that we need for our project. All of the ones we call are straightdforward -- they all help gain more information to calculate the GDP Growth. Keeping in mind GDP = Consumer Spending + Investment + Govt. Spending + Net Exports, collected many columns that related to these 4 categories. Chose T10Y2Y since its known to be a leading indicator for economic turns. We also log to stabilize variance and ensure we have stationarity for future models we might implement.  

In [2]:
SERIES = {
    "INDPRO":       {"freq": "M", "tf": "log_diff", "label": "Industrial Production"},
    "RSAFS":        {"freq": "M", "tf": "log_diff", "label": "Retail Sales"},
    "PAYEMS":       {"freq": "M", "tf": "log_diff", "label": "Nonfarm Payrolls"},
    "UNRATE":       {"freq": "M", "tf": "diff",     "label": "Unemployment Rate"},
    "HOUST":        {"freq": "M", "tf": "log_diff", "label": "Housing Starts"},
    "PCE":          {"freq": "M", "tf": "log_diff", "label": "Personal Consumption"},
    "DGORDER":      {"freq": "M", "tf": "log_diff", "label": "Durable Goods Orders"},
    "ICSA":         {"freq": "W", "tf": "log_diff", "label": "Initial Claims"},
    "T10Y2Y":       {"freq": "D", "tf": "level",    "label": "10Y-2Y Spread"},
}
TARGET = "A191RL1Q225SBEA"
BENCHMARK = "GDPNOW"

### Pulls a Fred Series into an actual Series in our environment. Works to isolate the API Call so we maintain a simple codebase. 

In [3]:
def fetch_series(code, start = START):
    s = fred.get_series(code, observation_start = START)
    s.name = code
    s.index = pd.to_datetime(s.index)
    return s.sort_index()

### Builds a dictonary of the series (raw data not the transformed version). Keeping it seperate from transformed in case we want to use different cases which doesn't use the transformed data (the logged version). 

In [5]:
def pull_all(series_map):
    return {code: fetch_series(code) for code in series_map}

raw = pull_all(SERIES)
target_raw = fetch_series(TARGET)
gdpnow_raw = fetch_series(BENCHMARK)

### Saving raw data in data/raw folder, helps so we can use it for offline if needed after first run. 

In [6]:
def save_raw(data, target, bench):
    raw_dir = DATA_DIR / "raw"
    raw_dir.mkdir(exist_ok=True)
    for code, s in data.items():
        s.to_csv(raw_dir / f"{code}.csv")
    target.to_csv(raw_dir / f"{TARGET}.csv")
    bench.to_csv(raw_dir / f"{BENCHMARK}.csv")

save_raw(raw, target_raw, gdpnow_raw)

### Summarizing to make sure all the columns are being pulled correctly, and there is a good amount of points for each column we can use. 

In [7]:
def coverage_row(code: str, s: pd.Series) -> dict:
    return {"code": code, "start": s.index.min().date(),
            "end": s.index.max().date(), "n": len(s)}

def coverage_table(data: dict, target: pd.Series, bench: pd.Series) -> pd.DataFrame:
    rows = [coverage_row(c, s) for c, s in data.items()]
    rows.append(coverage_row(TARGET, target))
    rows.append(coverage_row(BENCHMARK, bench))
    return pd.DataFrame(rows)

coverage_table(raw, target_raw, gdpnow_raw)

,code,start,end,n
0,INDPRO,1990-01-01,2026-03-01,435
1,RSAFS,1992-01-01,2026-02-01,410
2,PAYEMS,1990-01-01,2026-03-01,435
3,UNRATE,1990-01-01,2026-03-01,435
4,HOUST,1990-01-01,2026-01-01,433
5,PCE,1990-01-01,2026-02-01,434
6,DGORDER,1990-01-01,2026-02-01,434
7,ICSA,1990-01-06,2026-04-11,1893
8,T10Y2Y,1990-01-01,2026-04-17,9470
9,A191RL1Q225SBEA,1990-01-01,2025-10-01,144
